In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cvxpy as cp
from dotenv import load_dotenv
from tiingo import TiingoClient
from fredapi import Fred

load_dotenv()

# --- Universe ---
TICKERS = ['SPY', 'PLTR', 'NVDA', 'INTC', 'GLD', 'EEM']
ASSETS  = ['PLTR', 'NVDA', 'INTC', 'GLD', 'EEM']  # portfolio universe (excludes benchmark)
FREQ = '1hour'

# --- Date windows ---
FETCH_START = '2020-10-01'   # PLTR IPO'd 2020-09-30; gives a clean Nov-2020 start
FETCH_END   = '2026-04-30'

    # Witholding the training and testing windows for now
#TRAIN_START = '2020-11-01'
#TRAIN_END   = '2025-04-30'
#HOLD_START  = '2025-05-01'
#HOLD_END    = '2026-04-30'

# --- Local paths ---
DATA_DIR        = 'data'
RAW_CACHE       = f'{DATA_DIR}/intraday_1hr_raw.parquet'
CLEANED_CACHE = f'{DATA_DIR}/intraday_1hr_cleaned.parquet'

os.makedirs(DATA_DIR, exist_ok=True)

In [8]:
# === Reading data from Tiingo / Cache ===


if os.path.exists(RAW_CACHE):
    print(f"Loading raw cache: {RAW_CACHE}")
    raw = pd.read_parquet(RAW_CACHE)
else:
    print("Fetching data from Tiingo...")
    client = TiingoClient({'api_key': os.getenv('TIINGO_API_KEY')})

    dataframes = []

    for ticker in TICKERS:
        print(f"Fetching {ticker}...")

        df = client.get_dataframe(
            ticker, 
            startDate=FETCH_START, 
            endDate=FETCH_END, 
            frequency=FREQ,
            columns='open,high,low,close,volume')
            #afterHours=True, --- Not available on the standard endpoint; requires requests
            #forceFill=False)
        
        df['ticker'] = ticker
        df = df.reset_index(names='datetime')

        dataframes.append(df)

    raw = pd.concat(dataframes, ignore_index=True)

Fetching data from Tiingo...
Fetching SPY...
Fetching PLTR...
Fetching NVDA...
Fetching INTC...
Fetching GLD...
Fetching EEM...


In [10]:
print(raw['datetime'].min(), raw['datetime'].max())

print(raw.head())
print(raw.info())
raw.describe()

raw.to_parquet(RAW_CACHE, index=False)

2020-10-01 14:00:00+00:00 2026-04-30 19:00:00+00:00
                   datetime     open     high      low    close    volume  \
0 2020-10-01 14:00:00+00:00  336.605  337.435  335.560  336.680  113013.0   
1 2020-10-01 15:00:00+00:00  336.680  337.675  335.790  337.210   64050.0   
2 2020-10-01 16:00:00+00:00  337.195  337.420  336.235  337.145   74792.0   
3 2020-10-01 17:00:00+00:00  337.160  337.690  335.865  336.245   37771.0   
4 2020-10-01 18:00:00+00:00  336.245  337.195  334.995  336.955  141272.0   

  ticker  
0    SPY  
1    SPY  
2    SPY  
3    SPY  
4    SPY  
<class 'pandas.DataFrame'>
RangeIndex: 52416 entries, 0 to 52415
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype              
---  ------    --------------  -----              
 0   datetime  52416 non-null  datetime64[us, UTC]
 1   open      52416 non-null  float64            
 2   high      52416 non-null  float64            
 3   low       52416 non-null  float64            
 4   close     5